# Soft $c$-Transforms

Entropic regularization replaces the hard minimum in the $c$-transform by a log-sum-exp soft minimum,
$$
    \min_j^\varepsilon h_j=-\varepsilon\log\sum_j b_j e^{-h_j/\varepsilon}.
$$
This notebook uses the same translated quadratic functions as the hard-envelope figure and shows how increasing $\varepsilon$ smooths the lower envelope.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "notebooks-figures").exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "notebooks-figures"))

from figure_style import (
    RED, BLUE, VIOLET, ORANGE, GRAY, LIGHT_GRAY,
    DIRAC_MARKER_SIZE, setup_matplotlib, figure_dir, save_pdf,
    remove_axes, box_axes, padded_limits, interp_color, draw_point_clouds,
)

setup_matplotlib()

NAME = "sinkhorn-soft-c-transform-epsilon"
OUT = figure_dir(NAME)

The hard envelope is the limit as $\varepsilon\to0$.  We keep the same vertical scale in all panels to make the softening directly comparable.

In [ ]:
xs = np.array([-2.05, -0.75, 0.65, 1.85])
f = np.array([0.28, -0.30, 0.20, -0.08])
w = np.array([0.20, 0.32, 0.26, 0.22])
y = np.linspace(-3.0, 3.0, 700)
curves = 0.5 * (y[None, :] - xs[:, None])**2 - f[:, None]
hard = curves.min(axis=0)

def softmin(curves, eps):
    m = curves.min(axis=0)
    return m - eps*np.log((w[:, None] * np.exp(-(curves-m[None,:])/eps)).sum(axis=0))

eps_values = [(0.0, "hard"), (0.035, "eps-0p035"), (0.14, "eps-0p14"), (0.55, "eps-0p55")]
ymin = hard.min() - 0.28
ymax = np.percentile(curves, 72)

The gray parabolas are the individual competitors.  The colored curve is either the hard envelope or its soft-min approximation.

In [ ]:
def draw_panel(eps, path):
    fig, ax = plt.subplots(figsize=(2.55, 1.9))
    for i in range(len(xs)):
        ax.plot(y, curves[i], color=LIGHT_GRAY, lw=0.72, alpha=0.78)
    vals = hard if eps == 0 else softmin(curves, eps)
    color = VIOLET if eps == 0 else interp_color(min(eps/0.55, 1.0))
    ax.plot(y, vals, color=color, lw=1.55)
    ax.scatter(xs, -f, s=DIRAC_MARKER_SIZE, marker="o", color=RED, edgecolor="none", linewidth=0, zorder=4)
    ax.set_xlim(y.min(), y.max())
    ax.set_ylim(ymin, ymax)
    ax.set_xlabel(r"$y$")
    ax.set_ylabel(r"soft value")
    box_axes(ax)
    save_pdf(fig, OUT / f"{path}.pdf", pad_inches=0.055)
    plt.close(fig)

for eps, path in eps_values:
    draw_panel(eps, path)